# 🚀 LifeMate 2026: Chạy Planner & Safe Hash Guard trên Google Colab / VS Code

Notebook này hỗ trợ cả 2 chế độ:
1. **Chạy ngầm (Background Runner - GẬP MÁY ĐI NGỦ ĐƯỢC):** Chạy độc lập trên GPU cloud của Colab bằng `nohup`. Bạn có thể gập máy tính, tắt VS Code, GPU trên mây vẫn chạy liên tục cho đến khi xong đủ 300 mẫu.
2. **Chạy trực tiếp (Foreground) với Auto-Resume:** Nhìn thấy từng mẫu chạy trên màn hình. Nếu mất mạng hoặc laptop ngủ, chạy lại sẽ tự động tiếp tục (resume) từ mẫu dở dang.

In [5]:
# 0. Kết nối Google Drive (Bắt buộc để nạp dữ liệu data.zip và khôi phục checkpoint)
from google.colab import drive
drive.mount('/content/drive')
print('✅ Đã kết nối Google Drive thành công!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã kết nối Google Drive thành công!


In [6]:
# 1. Kiểm tra GPU và VRAM khả dụng
!nvidia-smi
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {device_name}')
    print(f'Total VRAM: {vram_gb:.2f} GB')

Sun Nov  8 05:16:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q -U vllm transformers peft accelerate datasets chunkformer

In [8]:
# 3. Clone hoặc cập nhật repository ZAIC_2026 trên nhánh planner/vuong
import os

# Luôn luôn xuất phát từ thư mục gốc /content (chống lỗi clone lồng nhau ZAIC_2026/ZAIC_2026)
%cd /content

if not os.path.exists('/content/ZAIC_2026/.git'):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git /content/ZAIC_2026

%cd /content/ZAIC_2026
!git checkout planner/vuong
!git pull origin planner/vuong
print('\nCommit mới nhất trên nhánh planner/vuong:')
!git log -n 1 --oneline

/content
/content/ZAIC_2026
Already on 'planner/vuong'
Your branch is up to date with 'origin/planner/vuong'.
From https://github.com/Bamboohoccode/ZAIC_2026
 * branch            planner/vuong -> FETCH_HEAD
Already up to date.

Commit mới nhất trên nhánh planner/vuong:
746552b (HEAD -> planner/vuong, origin/planner/vuong) [runner] them hard SIGALRM timeout va reset_prefix_cache de chong deadlock o mau dai


In [9]:
# 4. Thiết lập đường dẫn dữ liệu data/ (Tự động mount Drive và giải nén lên ổ cứng cục bộ /content)
import os, shutil, sys
from pathlib import Path

# 4.1. Tự động Mount Google Drive nếu chưa mount
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        print('🔗 Đang kết nối Google Drive...')
        drive.mount('/content/drive')
        print('✅ Đã kết nối Google Drive thành công!')
    except Exception as e:
        print(f'⚠️ Không thể kết nối Google Drive: {e}')

%cd /content/ZAIC_2026

# 4.2. Kiểm tra xem /content/data có sẵn chưa
has_local_data = os.path.exists('/content/data/train/train.jsonl') and os.path.exists('/content/data/train/zaic_tools/apps/banking.py')

if not has_local_data:
    # Tìm file zip data.zip trên Drive hoặc /content
    zip_candidates = [
        '/content/drive/MyDrive/data.zip',
        '/content/drive/MyDrive/ZAIC_2026/data.zip',
        '/content/drive/MyDrive/ZAIC_2026_runs/data.zip',
        '/content/drive/MyDrive/ZAIC/data.zip',
        '/content/data.zip',
    ]
    found_zip = next((z for z in zip_candidates if os.path.exists(z)), None)
    if not found_zip and os.path.exists('/content/drive/MyDrive'):
        for f in os.listdir('/content/drive/MyDrive'):
            if f.lower().endswith('.zip') and ('data' in f.lower() or 'train' in f.lower()):
                found_zip = os.path.join('/content/drive/MyDrive', f)
                break

    if found_zip:
        print(f'📦 Đang giải nén dữ liệu từ {found_zip} sang ổ cứng cục bộ /content/ ... (mất ~15 giây)')
        !unzip -q -o "{found_zip}" -d /content/
    elif os.path.exists('/content/drive/MyDrive/data/train'):
        print('📁 Đang sao chép thư mục data/train từ Google Drive sang /content/data/train ...')
        os.makedirs('/content/data', exist_ok=True)
        !cp -r /content/drive/MyDrive/data/train /content/data/train
    else:
        print('⚠️ Không tìm thấy data.zip trên Drive hoặc /content!')

# 4.3. Đảm bảo /content/data/train tồn tại (kể cả khi zip bung ra /content/train)
if not os.path.exists('/content/data/train') and os.path.exists('/content/train'):
    os.makedirs('/content/data', exist_ok=True)
    !mv /content/train /content/data/train

os.makedirs('data', exist_ok=True)
if os.path.exists('/content/data/train'):
    !rm -rf data/train
    !ln -s /content/data/train data/train
if os.path.exists('/content/data/public_test'):
    !rm -rf data/public_test
    !ln -s /content/data/public_test data/public_test

# 4.4. KIỂM TRA TỰ ĐỘNG: Thêm đường dẫn tuyệt đối vào sys.path để import chuẩn xác
for p in [os.path.abspath('data/train'), '/content/data/train', '/content/ZAIC_2026/data/train']:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

try:
    from zaic_tools import ToolEnv
    print('✅ Dữ liệu data/train và module zaic_tools.apps: HOẠT ĐỘNG HOÀN HẢO!')
    !ls -lh data/train | head -n 6
except Exception as e:
    print('❌ LỖI DỮ LIỆU:', e)
    print('Vui lòng kiểm tra file data.zip có chứa thư mục zaic_tools/apps hay không!')

/content/ZAIC_2026
📦 Đang giải nén dữ liệu từ /content/drive/MyDrive/data.zip sang ổ cứng cục bộ /content/ ... (mất ~15 giây)
✅ Dữ liệu data/train và module zaic_tools.apps: HOẠT ĐỘNG HOÀN HẢO!
lrwxrwxrwx 1 root root 19 Nov  8 05:16 data/train -> /content/data/train


### 🎯 LỰA CHỌN CÁCH CHẠY PIPELINE:
- **Cách 5A (Khuyên dùng khi đi ngủ):** Chạy ngầm trong background với `nohup`. Bạn có thể **gập máy tính lại**, GPU Colab vẫn chạy tiếp đến khi xong!
- **Cách 5B (Khi đang ngồi máy):** Chạy trực tiếp trên màn hình có kèm cờ `--resume` (nếu lỡ tắt máy, chạy lại sẽ tự làm tiếp từ mẫu vừa dừng).

In [10]:
# 5A. [KHUYÊN DÙNG] CHẠY NGẦM BACKGROUND (GẬP MÁY / SLEEP LAPTOP THOẢI MÁI)
%cd /content/ZAIC_2026
# Dọn dẹp tiến trình cũ để giải phóng 100% VRAM GPU
!pkill -9 -f 'lifemate.planner.runner' || true
!pkill -9 -f 'lifemate.pipeline' || true
import os, time, shutil
time.sleep(2)
os.makedirs('runs', exist_ok=True)

# Tự động khôi phục checkpoint từ Google Drive nếu máy ảo vừa được tạo mới
drive_checkpoint = '/content/drive/MyDrive/ZAIC_2026_runs/dev_v4_guarded.jsonl'
if os.path.exists(drive_checkpoint) and not os.path.exists('runs/dev_v4_guarded.jsonl'):
    shutil.copy(drive_checkpoint, 'runs/dev_v4_guarded.jsonl')
    with open('runs/dev_v4_guarded.jsonl', 'r', encoding='utf-8') as f:
        n_resumed = sum(1 for line in f if line.strip())
    print(f'🔄 Đã tìm thấy {n_resumed} mẫu đã chạy trên Google Drive! Tự động nạp vào runs/ để tiếp tục.')

# Chạy ngầm bằng nohup (sử dụng runner chuyên dụng của Vương trong lifemate/planner/)
!nohup python -m lifemate.planner.runner \
    --root data \
    --split train \
    --text \
    --llm Qwen/Qwen3-4B-Instruct-2507 \
    --mapper none \
    --gpu-mem 0.90 \
    --max-ctx 8192 \
    --verify-rounds 0 \
    --prompt prompts/gepa/vuong/planner_v4.txt \
    --ids eval/splits/dev_ids.txt \
    --resume \
    --out runs/dev_v4_guarded.jsonl > runs/dev_v4_guarded.nohup.log 2>&1 &

print('🚀 Tiến trình đã được đưa vào chạy ngầm trên GPU Cloud (chuẩn --gpu-mem 0.90 như Kaggle)!')
print('Bạn có thể gập máy tính lại hoặc theo dõi tiến độ ở Cell 5C bên dưới.')

/content/ZAIC_2026
^C
^C
🔄 Đã tìm thấy 300 mẫu đã chạy trên Google Drive! Tự động nạp vào runs/ để tiếp tục.
🚀 Tiến trình đã được đưa vào chạy ngầm trên GPU Cloud (chuẩn --gpu-mem 0.90 như Kaggle)!
Bạn có thể gập máy tính lại hoặc theo dõi tiến độ ở Cell 5C bên dưới.


In [ ]:
# 5B. (Tuỳ chọn) CHẠY TRỰC TIẾP TRÊN MÀN HÌNH (CÓ TỰ ĐỘNG RESUME TIẾP TỤC)
# %cd /content/ZAIC_2026
# import os
# os.makedirs('runs', exist_ok=True)
#
# !python -m lifemate.planner.runner \
#     --root data \
#     --split train \
#     --text \
#     --llm Qwen/Qwen3-4B-Instruct-2507 \
#     --mapper none \
#     --gpu-mem 0.90 \
#     --max-ctx 8192 \
#     --verify-rounds 0 \
#     --prompt prompts/gepa/vuong/planner_v4.txt \
#     --ids eval/splits/dev_ids.txt \
#     --resume \
#     --out runs/dev_v4_guarded.jsonl

In [ ]:
# 5C. THEO DÕI TIẾN ĐỘ THỜI GIAN THỰC (Tự động cập nhật liên tục)
%cd /content/ZAIC_2026
import time, os
from IPython.display import clear_output

out_path = 'runs/dev_v4_guarded.jsonl'
log_path = 'runs/dev_v4_guarded.jsonl.log'
nohup_log = 'runs/dev_v4_guarded.nohup.log'

loop_count = 0
while True:  # Cập nhật liên tục không giới hạn 10 phút, chỉ dừng khi xong hoặc có lỗi
    clear_output(wait=True)
    loop_count += 1
    done_count = 0
    if os.path.exists(out_path):
        with open(out_path, 'r', encoding='utf-8') as f:
            done_count = sum(1 for line in f if line.strip())
    
    pids = os.popen("pgrep -f 'lifemate.planner.runner' || pgrep -f 'lifemate.pipeline'").read().strip().split()
    print(f'📊 TIẾN ĐỘ: Đã xong {done_count}/300 mẫu dev ({done_count/300*100:.1f}%)')
    print('=' * 65)
    
    # Tự động đồng bộ ngay sang Google Drive để bạn theo dõi trên điện thoại/Drive
    if os.path.exists('/content/drive/MyDrive') and done_count > 0:
        drive_dir = '/content/drive/MyDrive/ZAIC_2026_runs'
        os.makedirs(drive_dir, exist_ok=True)
        os.system(f'cp -f {out_path} {drive_dir}/ 2>/dev/null')
        os.system(f'cp -f {log_path} {drive_dir}/ 2>/dev/null')
    
    if os.path.exists(log_path) and os.path.getsize(log_path) > 0:
        !tail -n 12 runs/dev_v4_guarded.jsonl.log
    elif os.path.exists(nohup_log):
        print('⏳ Đang khởi động và làm nóng mô hình vLLM (Torch compile & CUDA graphs)...')
        print('Quá trình làm nóng này mất khoảng 2-3 phút trước khi chạy mẫu đầu tiên:')
        !tail -n 8 runs/dev_v4_guarded.nohup.log
    
    if done_count >= 300:
        print('\n🎉 TIẾN TRÌNH ĐÃ HOÀN THÀNH XONG ĐỦ 300 MẪU! Hãy dừng cell này và chạy Cell 6 để xem điểm số.')
        break
    elif pids:
        print(f'\n⚡ Tiến trình GPU đang xử lý (PID: {pids[0]})... [Cập nhật #{loop_count}]')
        time.sleep(5)
    else:
        if done_count > 0:
            print(f'\n⏹️ Tiến trình đã dừng lại ở {done_count}/300 mẫu.')
            if os.path.exists(nohup_log):
                print('--- Chi tiết đuôi log nohup ---')
                !tail -n 15 runs/dev_v4_guarded.nohup.log
            print('Có thể chạy lại Cell 5A để tự động resume tiếp tục mẫu kế tiếp.')
        else:
            print('\n⏹️ Chưa có tiến trình nào đang chạy.')
        break


📊 TIẾN ĐỘ: Đã xong 281/300 mẫu dev (93.7%)
[03:15:45]     -> [LLM Output in 8.7s] calls=['home_automation.delete_rule'] | preview: <tool_call>
[03:15:45]     -> [Tool Exec] home_automation.delete_rule in 0.00s -> OK
[03:15:45]   [Step 4/8] time_left=85.0s
[03:15:45]     -> [LLM Chat] msgs=27 input_tokens=7067 calling vLLM...
[03:15:49]     -> [LLM Output in 3.8s] calls=['chat_ott.send_message'] | preview: <tool_call>
[03:15:49]     -> [Tool Exec] chat_ott.send_message in 0.00s -> ERR: Giá trị ['chat_id'] chưa xuất hiện trong
[03:15:49]   [Step 5/8] time_left=81.0s
[03:15:49]     -> [LLM Chat] msgs=28 input_tokens=7160 calling vLLM...
[03:15:58]     -> [LLM Output in 8.8s] calls=['chat_ott.list_chats'] | preview: <tool_call>
[03:15:58]     -> [Tool Exec] chat_ott.list_chats in 0.00s -> OK
[03:15:58]   [Step 6/8] time_left=72.1s
[03:15:59]     -> [LLM Chat] msgs=25 input_tokens=7049 calling vLLM...

⚡ Tiến trình GPU đang xử lý (PID: 17794)... [Cập nhật #895]


In [12]:
!nvidia-smi

Sun Nov  8 05:17:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# !pkill -9 -f 'lifemate.pipeline'

In [ ]:
# 6. Chấm điểm và phân tích lỗi chi tiết với eval/scorer.py
%cd /content/ZAIC_2026
import os

sub_file = 'runs/dev_v4_guarded.jsonl'
if not os.path.exists(sub_file) or os.path.getsize(sub_file) == 0:
    print('⚠️ Chưa có dữ liệu kết quả để chấm điểm!')
    print('Lý do: Tiến trình GPU ở Cell 5 đang khởi động vLLM hoặc đang chạy những mẫu đầu tiên.')
    print('👉 Hãy đợi Cell 5C hoàn thành ít nhất 1 mẫu rồi chạy lại cell này để chấm điểm!')
else:
    with open(sub_file, 'r', encoding='utf-8') as f:
        n_samples = sum(1 for line in f if line.strip())
    print(f'Đang chấm điểm trên {n_samples}/300 mẫu đã hoàn thành...')
    !python eval/scorer.py --sub runs/dev_v4_guarded.jsonl


/content/ZAIC_2026
Đang chấm điểm trên 16/300 mẫu đã hoàn thành...

===== KẾT QUẢ ĐÁNH GIÁ (N=16) =====
Mean Score: 0.5908
  act  : 0.4963 (n=13)
  empty: 1.0000 (n=3)

Phân loại lỗi:
  HASH_MISMATCH       : 4 (25.0%)
  WRONG_TOOL_OR_ARGS  : 3 (18.8%)
  EXTRA_CALL          : 1 (6.2%)


In [ ]:
# 7. Tự động sao lưu file kết quả và run log vào Google Drive
%cd /content/ZAIC_2026
import os, shutil
from pathlib import Path

if os.path.exists('/content/drive/MyDrive'):
    backup_dir = Path('/content/drive/MyDrive/ZAIC_2026_runs')
    backup_dir.mkdir(parents=True, exist_ok=True)
    for fname in ['dev_v4_guarded.jsonl', 'dev_v4_guarded.jsonl.log', 'dev_v4_guarded.jsonl.trace.jsonl']:
        fpath = Path('runs') / fname
        if fpath.exists():
            shutil.copy(fpath, backup_dir / fname)
            print(f'✅ Đã lưu: {fname} -> {backup_dir}')
    print(f'Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: {backup_dir}')
else:
    print('File kết quả hiện nằm trực tiếp tại: runs/dev_v4_guarded.jsonl')

/content/ZAIC_2026
✅ Đã lưu: dev_v4_guarded.jsonl -> /content/drive/MyDrive/ZAIC_2026_runs
✅ Đã lưu: dev_v4_guarded.jsonl.log -> /content/drive/MyDrive/ZAIC_2026_runs
✅ Đã lưu: dev_v4_guarded.jsonl.trace.jsonl -> /content/drive/MyDrive/ZAIC_2026_runs
Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: /content/drive/MyDrive/ZAIC_2026_runs
